# Evaluating Model Outputs

We can evaluate a model's confidence in its results by using perplexity. Perplexity is a measure of uncertainty that can be calculated by exponentiating the negative of the average of the logprobs. 

+ Perplexity can be used to assess the result of an individual model run.
+ It can also be used to compare the relative confidence of results between model runs. 

Low perplexity or high confidence does not guarantee accuracy, but it can be a helpful signal when paired with other evaluation metrics. 

In [1]:
%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../../05_src/.secrets
import sys
sys.path.append('../../05_src/')

In [2]:
from utils.clients import get_client
from IPython.display import display, Markdown
import numpy as np

client = get_client()

In [3]:
prompts = [
    # Low perplexity: Clear topic, common structure, highly predictable vocabulary
    "Explain how photosynthesis works in simple terms.",
    # Medium preplexity: Narrative freedom, but familiar theme and constraints.
    "Write a short story about a traveler who realizes the journey mattered more than the destination.",
    # High perplexity: Abstract concept, creative freedom, unpredictable vocabulary
    "Describe the taste of a color that only exists for one second at dusk, using metaphors from mathematics and weather."
]

In [4]:
def get_completion(
    input: list[dict[str, str]],
    model: str = "gpt-4o-mini",
    max_tokens=500,
    temperature=0,
    tools=None,
    logprobs=None,  # whether to return log probabilities of the output tokens or not. If true, returns the log probabilities of each output token returned in the content of message..
    top_logprobs=None,
) -> str:
    params = {
        "model": model,
        "input": input,
        "max_output_tokens": max_tokens,
        "temperature": temperature,
        "tools": tools,
        "include": ["message.output_text.logprobs"] if logprobs else [],
        "top_logprobs": top_logprobs,
    }
    if tools:
        params["tools"] = tools

    completion = client.responses.create(**params)
    return completion

In [5]:
for k, prompt in enumerate(prompts):
    API_RESPONSE = get_completion(
        [{"role": "user", "content": prompt}],
        model="gpt-4o-mini",
        logprobs=True,
    )
    token_data = API_RESPONSE.output[0].content[0].logprobs
    response_text = API_RESPONSE.output[0].content[0].text
    lp_values = [t.logprob for t in token_data]
    perplexity_score = np.exp(-np.mean(lp_values))

    rows = [
        f"| `{t.token.replace('|', chr(9474)).replace(chr(10), '↵').replace(chr(96), chr(39))}` "
        f"| {t.logprob:.4f} | {np.exp(t.logprob)*100:.1f}% |"
        for t in token_data
    ]
    table = "\n".join([
        "| Token | Logprob | Linear Prob |",
        "|:------|--------:|------------:|",
    ] + rows)

    display(Markdown(f"### Prompt {k+1}\n_{prompt}_"))
    display(Markdown(f"**Response:**\n\n{response_text}"))
    display(Markdown(table))
    display(Markdown(f"**Perplexity:** `{perplexity_score:.2f}`\n\n---"))

### Prompt 1
_Explain how photosynthesis works in simple terms._

**Response:**

Photosynthesis is the process that plants, algae, and some bacteria use to make their own food. Here’s how it works in simple terms:

1. **Sunlight**: Plants take in sunlight using a green pigment called chlorophyll, which is found in their leaves.

2. **Water**: They absorb water from the soil through their roots.

3. **Carbon Dioxide**: Plants take in carbon dioxide from the air through tiny openings in their leaves called stomata.

4. **Making Food**: Using the energy from sunlight, plants combine water and carbon dioxide to create glucose (a type of sugar) and oxygen. The glucose is used as food for energy and growth.

5. **Oxygen Release**: The oxygen produced during this process is released back into the air, which is essential for us and other living beings to breathe.

In summary, photosynthesis is how plants turn sunlight, water, and carbon dioxide into food and oxygen!

| Token | Logprob | Linear Prob |
|:------|--------:|------------:|
| `Photos` | -0.0793 | 92.4% |
| `ynthesis` | 0.0000 | 100.0% |
| ` is` | -0.0000 | 100.0% |
| ` the` | -0.0467 | 95.4% |
| ` process` | -0.0041 | 99.6% |
| ` that` | -0.2280 | 79.6% |
| ` plants` | -0.0044 | 99.6% |
| `,` | -0.5767 | 56.2% |
| ` algae` | -0.0296 | 97.1% |
| `,` | 0.0000 | 100.0% |
| ` and` | 0.0000 | 100.0% |
| ` some` | -0.0000 | 100.0% |
| ` bacteria` | -0.0001 | 100.0% |
| ` use` | -0.0000 | 100.0% |
| ` to` | 0.0000 | 100.0% |
| ` make` | -0.1630 | 85.0% |
| ` their` | -0.0067 | 99.3% |
| ` own` | -0.0620 | 94.0% |
| ` food` | -0.0000 | 100.0% |
| `.` | -0.5784 | 56.1% |
| ` Here` | -0.2049 | 81.5% |
| `’s` | -0.0001 | 100.0% |
| ` how` | -0.1002 | 90.5% |
| ` it` | 0.0000 | 100.0% |
| ` works` | -0.0000 | 100.0% |
| ` in` | -0.0238 | 97.6% |
| ` simple` | -0.0001 | 100.0% |
| ` terms` | -0.0015 | 99.8% |
| `:↵↵` | -0.0000 | 100.0% |
| `1` | -0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | -0.0000 | 100.0% |
| `Sun` | -0.7149 | 48.9% |
| `light` | -0.0002 | 100.0% |
| `**` | -0.1053 | 90.0% |
| `:` | -0.0000 | 100.0% |
| ` Plants` | -0.0010 | 99.9% |
| ` take` | -0.8444 | 43.0% |
| ` in` | -0.0005 | 99.9% |
| ` sunlight` | -0.0190 | 98.1% |
| ` using` | -0.4959 | 60.9% |
| ` a` | -0.1340 | 87.5% |
| ` green` | -0.0366 | 96.4% |
| ` pigment` | -0.0021 | 99.8% |
| ` called` | -0.0486 | 95.3% |
| ` chlor` | -0.0000 | 100.0% |
| `ophyll` | 0.0000 | 100.0% |
| `,` | -0.5888 | 55.5% |
| ` which` | -0.6942 | 49.9% |
| ` is` | -0.0001 | 100.0% |
| ` found` | -0.3869 | 67.9% |
| ` in` | -0.0130 | 98.7% |
| ` their` | -0.0023 | 99.8% |
| ` leaves` | -0.0000 | 100.0% |
| `.↵↵` | -0.0030 | 99.7% |
| `2` | 0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | 0.0000 | 100.0% |
| `Water` | -0.3304 | 71.9% |
| `**` | -0.3157 | 72.9% |
| `:` | 0.0000 | 100.0% |
| ` They` | -0.7692 | 46.3% |
| ` absorb` | -0.0434 | 95.7% |
| ` water` | -0.0000 | 100.0% |
| ` from` | -0.0753 | 92.7% |
| ` the` | -0.0001 | 100.0% |
| ` soil` | -0.0182 | 98.2% |
| ` through` | -0.0000 | 100.0% |
| ` their` | -0.0000 | 100.0% |
| ` roots` | -0.0000 | 100.0% |
| `.↵↵` | -0.0006 | 99.9% |
| `3` | 0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | 0.0000 | 100.0% |
| `Carbon` | -0.0005 | 100.0% |
| ` D` | -0.0067 | 99.3% |
| `ioxide` | -0.0000 | 100.0% |
| `**` | -0.0003 | 100.0% |
| `:` | 0.0000 | 100.0% |
| ` Plants` | -0.0103 | 99.0% |
| ` take` | -0.2845 | 75.2% |
| ` in` | -0.0007 | 99.9% |
| ` carbon` | -0.0041 | 99.6% |
| ` dioxide` | 0.0000 | 100.0% |
| ` from` | -0.0420 | 95.9% |
| ` the` | -0.0000 | 100.0% |
| ` air` | -0.0000 | 100.0% |
| ` through` | -0.0002 | 100.0% |
| ` tiny` | -0.1004 | 90.5% |
| ` openings` | -0.0113 | 98.9% |
| ` in` | -0.0189 | 98.1% |
| ` their` | -0.0003 | 100.0% |
| ` leaves` | 0.0000 | 100.0% |
| ` called` | -0.0068 | 99.3% |
| ` stom` | -0.0001 | 100.0% |
| `ata` | -0.0000 | 100.0% |
| `.↵↵` | -0.0000 | 100.0% |
| `4` | -0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | 0.0000 | 100.0% |
| `Making` | -1.0087 | 36.5% |
| ` Food` | -0.0044 | 99.6% |
| `**` | -0.0001 | 100.0% |
| `:` | -0.0001 | 100.0% |
| ` Using` | -0.0369 | 96.4% |
| ` the` | -0.4079 | 66.5% |
| ` energy` | -0.2530 | 77.6% |
| ` from` | -0.0000 | 100.0% |
| ` sunlight` | -0.0041 | 99.6% |
| `,` | -0.0000 | 100.0% |
| ` plants` | -0.0117 | 98.8% |
| ` combine` | -0.2054 | 81.4% |
| ` water` | -0.2083 | 81.2% |
| ` and` | -0.0001 | 100.0% |
| ` carbon` | 0.0000 | 100.0% |
| ` dioxide` | 0.0000 | 100.0% |
| ` to` | -0.0031 | 99.7% |
| ` create` | -0.2900 | 74.8% |
| ` glucose` | -0.0219 | 97.8% |
| ` (` | -0.0382 | 96.2% |
| `a` | -0.0003 | 100.0% |
| ` type` | -0.0120 | 98.8% |
| ` of` | 0.0000 | 100.0% |
| ` sugar` | -0.0000 | 100.0% |
| `)` | -0.1115 | 89.5% |
| ` and` | -0.0020 | 99.8% |
| ` oxygen` | -0.0006 | 99.9% |
| `.` | -0.1799 | 83.5% |
| ` The` | -0.5691 | 56.6% |
| ` glucose` | -0.2593 | 77.2% |
| ` is` | -0.5947 | 55.2% |
| ` used` | -0.4054 | 66.7% |
| ` as` | -0.1968 | 82.1% |
| ` food` | -0.0299 | 97.1% |
| ` for` | -0.0679 | 93.4% |
| ` energy` | -0.4645 | 62.8% |
| ` and` | -0.0188 | 98.1% |
| ` growth` | -0.0000 | 100.0% |
| `.↵↵` | -0.3490 | 70.5% |
| `5` | -0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | 0.0000 | 100.0% |
| `O` | -0.1877 | 82.9% |
| `xygen` | -0.0000 | 100.0% |
| ` Release` | -0.0289 | 97.2% |
| `**` | -0.0000 | 100.0% |
| `:` | 0.0000 | 100.0% |
| ` The` | -0.4158 | 66.0% |
| ` oxygen` | -0.1839 | 83.2% |
| ` produced` | -0.0212 | 97.9% |
| ` during` | -0.1507 | 86.0% |
| ` this` | -0.0260 | 97.4% |
| ` process` | -0.0000 | 100.0% |
| ` is` | -0.0002 | 100.0% |
| ` released` | -0.0024 | 99.8% |
| ` back` | -0.5759 | 56.2% |
| ` into` | 0.0000 | 100.0% |
| ` the` | 0.0000 | 100.0% |
| ` air` | -0.0142 | 98.6% |
| `,` | -0.0034 | 99.7% |
| ` which` | -0.0012 | 99.9% |
| ` is` | -0.0205 | 98.0% |
| ` essential` | -0.9232 | 39.7% |
| ` for` | 0.0000 | 100.0% |
| ` us` | -0.8155 | 44.2% |
| ` and` | -0.1026 | 90.2% |
| ` other` | -0.0634 | 93.9% |
| ` living` | -0.0813 | 92.2% |
| ` beings` | -0.4794 | 61.9% |
| ` to` | -0.0299 | 97.1% |
| ` breathe` | -0.0000 | 100.0% |
| `.↵↵` | -0.0007 | 99.9% |
| `In` | -0.3878 | 67.9% |
| ` summary` | -0.2571 | 77.3% |
| `,` | -0.0060 | 99.4% |
| ` photos` | -0.1198 | 88.7% |
| `ynthesis` | 0.0000 | 100.0% |
| ` is` | -1.0730 | 34.2% |
| ` how` | -0.0816 | 92.2% |
| ` plants` | -0.0001 | 100.0% |
| ` turn` | -0.3112 | 73.3% |
| ` sunlight` | -0.0068 | 99.3% |
| `,` | -0.0298 | 97.1% |
| ` water` | -0.0002 | 100.0% |
| `,` | -0.0000 | 100.0% |
| ` and` | 0.0000 | 100.0% |
| ` carbon` | -0.0182 | 98.2% |
| ` dioxide` | 0.0000 | 100.0% |
| ` into` | 0.0000 | 100.0% |
| ` food` | -0.0056 | 99.4% |
| ` and` | -0.1727 | 84.1% |
| ` oxygen` | -0.0013 | 99.9% |
| `!` | -0.2149 | 80.7% |

**Perplexity:** `1.11`

---

### Prompt 2
_Write a short story about a traveler who realizes the journey mattered more than the destination._

...

**Perplexity:** `1.47`

---

### Prompt 3
_Describe the taste of a color that only exists for one second at dusk, using metaphors from mathematics and weather._

**Response:**

Imagine a color that tastes like the fleeting moment when twilight kisses the horizon—a blend of soft lavender and deep indigo, like the gentle curve of a parabolic arc just before it meets the ground. It’s the sweet tang of a cool breeze, whispering secrets of impending rain, as if the atmosphere is holding its breath in anticipation.

This color tastes like the delicate balance of a probability curve, where the odds of savoring its essence are slim, yet infinitely precious. It’s the ephemeral warmth of a sunbeam breaking through a cloud, a fleeting moment of clarity in a chaotic storm, where the flavors of hope and nostalgia collide like two intersecting lines, creating a point of intersection that is both beautiful and transient.

As it dissolves into the night, it leaves a lingering sensation, like the last drop of dew on a blade of grass, a reminder of the delicate equations of time and light that govern our perception. It’s a taste that dances on the tongue, a fleeting theorem of dusk that vanishes before it can be fully grasped, leaving only the echo of its brilliance behind.

| Token | Logprob | Linear Prob |
|:------|--------:|------------:|
| `Imagine` | -0.3913 | 67.6% |
| ` a` | -0.5276 | 59.0% |
| ` color` | -0.4897 | 61.3% |
| ` that` | -0.0093 | 99.1% |
| ` tastes` | -1.4597 | 23.2% |
| ` like` | -0.0003 | 100.0% |
| ` the` | -0.2203 | 80.2% |
| ` fleeting` | -0.1759 | 83.9% |
| ` moment` | -0.2845 | 75.2% |
| ` when` | -0.1918 | 82.5% |
| ` twilight` | -0.6918 | 50.1% |
| ` kisses` | -1.8035 | 16.5% |
| ` the` | -0.0497 | 95.2% |
| ` horizon` | -0.0963 | 90.8% |
| `—a` | -0.7893 | 45.4% |
| ` blend` | -1.6456 | 19.3% |
| ` of` | -0.0243 | 97.6% |
| ` soft` | -1.6607 | 19.0% |
| ` lavender` | -0.5707 | 56.5% |
| ` and` | -0.0168 | 98.3% |
| ` deep` | -0.2785 | 75.7% |
| ` ind` | -0.1349 | 87.4% |
| `igo` | -0.0000 | 100.0% |
| `,` | -0.3303 | 71.9% |
| ` like` | -1.0760 | 34.1% |
| ` the` | -0.4080 | 66.5% |
| ` gentle` | -0.5459 | 57.9% |
| ` curve` | -0.3672 | 69.3% |
| ` of` | -0.0000 | 100.0% |
| ` a` | -0.0431 | 95.8% |
| ` par` | -0.6079 | 54.4% |
| `abolic` | -0.0016 | 99.8% |
| ` arc` | -0.0521 | 94.9% |
| ` just` | -1.1944 | 30.3% |
| ` before` | -0.0131 | 98.7% |
| ` it` | -0.0153 | 98.5% |
| ` meets` | -1.3963 | 24.8% |
| ` the` | -0.2302 | 79.4% |
| ` ground` | -0.5757 | 56.2% |
| `.` | -0.0434 | 95.8% |
| ` It` | -0.6960 | 49.9% |
| `’s` | -0.4543 | 63.5% |
| ` the` | -0.6347 | 53.0% |
| ` sweet` | -1.5733 | 20.7% |
| ` tang` | -0.2892 | 74.9% |
| ` of` | -0.0001 | 100.0% |
| ` a` | -0.6489 | 52.3% |
| ` cool` | -1.3814 | 25.1% |
| ` breeze` | -0.0214 | 97.9% |
| `,` | -1.1845 | 30.6% |
| ` whisper` | -1.3691 | 25.4% |
| `ing` | -0.0000 | 100.0% |
| ` secrets` | -0.7631 | 46.6% |
| ` of` | -0.1689 | 84.5% |
| ` impending` | -1.5811 | 20.6% |
| ` rain` | -0.3523 | 70.3% |
| `,` | -0.0813 | 92.2% |
| ` as` | -1.4164 | 24.3% |
| ` if` | -0.1458 | 86.4% |
| ` the` | -0.4394 | 64.4% |
| ` atmosphere` | -1.0969 | 33.4% |
| ` is` | -0.6465 | 52.4% |
| ` holding` | -0.7766 | 46.0% |
| ` its` | -0.0033 | 99.7% |
| ` breath` | -0.0000 | 100.0% |
| ` in` | -0.6472 | 52.3% |
| ` anticipation` | -0.7034 | 49.5% |
| `.↵↵` | -0.3964 | 67.3% |
| `This` | -0.0324 | 96.8% |
| ` color` | -0.4665 | 62.7% |
| ` tastes` | -0.9121 | 40.2% |
| ` like` | -0.0059 | 99.4% |
| ` the` | -0.3496 | 70.5% |
| ` delicate` | -2.0047 | 13.5% |
| ` balance` | -0.0560 | 94.6% |
| ` of` | -0.0255 | 97.5% |
| ` a` | -0.5736 | 56.3% |
| ` probability` | -1.9340 | 14.5% |
| ` curve` | -1.1467 | 31.8% |
| `,` | -0.0813 | 92.2% |
| ` where` | -0.0644 | 93.8% |
| ` the` | -0.7496 | 47.3% |
| ` odds` | -1.8792 | 15.3% |
| ` of` | -0.3854 | 68.0% |
| ` savor` | -1.2409 | 28.9% |
| `ing` | -0.0001 | 100.0% |
| ` its` | -0.5440 | 58.0% |
| ` essence` | -0.0770 | 92.6% |
| ` are` | -0.2401 | 78.7% |
| ` slim` | -1.1378 | 32.1% |
| `,` | -0.8554 | 42.5% |
| ` yet` | -0.0384 | 96.2% |
| ` infinitely` | -1.3466 | 26.0% |
| ` precious` | -1.7660 | 17.1% |
| `.` | -0.0293 | 97.1% |
| ` It` | -0.3071 | 73.6% |
| `’s` | -0.3911 | 67.6% |
| ` the` | -0.7891 | 45.4% |
| ` ephemeral` | -2.0419 | 13.0% |
| ` warmth` | -2.0267 | 13.2% |
| ` of` | -0.0143 | 98.6% |
| ` a` | -0.2862 | 75.1% |
| ` sun` | -0.3940 | 67.4% |
| `beam` | -0.0170 | 98.3% |
| ` breaking` | -1.4847 | 22.7% |
| ` through` | -0.0029 | 99.7% |
| ` a` | -0.6449 | 52.5% |
| ` cloud` | -0.9473 | 38.8% |
| `,` | -0.1577 | 85.4% |
| ` a` | -1.0826 | 33.9% |
| ` fleeting` | -0.5769 | 56.2% |
| ` moment` | -1.9855 | 13.7% |
| ` of` | -1.0887 | 33.7% |
| ` clarity` | -0.7782 | 45.9% |
| ` in` | -0.6054 | 54.6% |
| ` a` | -0.3654 | 69.4% |
| ` chaotic` | -0.4965 | 60.9% |
| ` storm` | -0.1243 | 88.3% |
| `,` | -0.5181 | 59.6% |
| ` where` | -1.4293 | 23.9% |
| ` the` | -1.1906 | 30.4% |
| ` flavors` | -1.0943 | 33.5% |
| ` of` | -1.4570 | 23.3% |
| ` hope` | -1.3878 | 25.0% |
| ` and` | -0.0119 | 98.8% |
| ` nostalgia` | -0.4054 | 66.7% |
| ` collide` | -1.3227 | 26.6% |
| ` like` | -0.8871 | 41.2% |
| ` two` | -1.5141 | 22.0% |
| ` intersect` | -0.2775 | 75.8% |
| `ing` | -0.0000 | 100.0% |
| ` lines` | -0.0016 | 99.8% |
| `,` | -1.0368 | 35.5% |
| ` creating` | -0.5465 | 57.9% |
| ` a` | -0.2253 | 79.8% |
| ` point` | -0.4098 | 66.4% |
| ` of` | -0.0140 | 98.6% |
| ` intersection` | -1.2179 | 29.6% |
| ` that` | -0.0999 | 90.5% |
| ` is` | -1.5070 | 22.2% |
| ` both` | -0.1014 | 90.4% |
| ` beautiful` | -1.1249 | 32.5% |
| ` and` | -0.0000 | 100.0% |
| ` transient` | -0.5660 | 56.8% |
| `.↵↵` | -0.0487 | 95.2% |
| `As` | -0.4935 | 61.0% |
| ` it` | -0.5025 | 60.5% |
| ` dissol` | -1.3447 | 26.1% |
| `ves` | 0.0000 | 100.0% |
| ` into` | -0.6474 | 52.3% |
| ` the` | -0.1099 | 89.6% |
| ` night` | -0.7683 | 46.4% |
| `,` | -0.0025 | 99.8% |
| ` it` | -0.8311 | 43.6% |
| ` leaves` | -0.0324 | 96.8% |
| ` a` | -0.7066 | 49.3% |
| ` lingering` | -0.8702 | 41.9% |
| ` sensation` | -0.9169 | 40.0% |
| `,` | -1.0596 | 34.7% |
| ` like` | -0.6520 | 52.1% |
| ` the` | -0.0652 | 93.7% |
| ` last` | -1.5816 | 20.6% |
| ` drop` | -0.9070 | 40.4% |
| ` of` | -0.0025 | 99.8% |
| ` dew` | -0.3468 | 70.7% |
| ` on` | -0.5802 | 56.0% |
| ` a` | -0.2010 | 81.8% |
| ` blade` | -1.5500 | 21.2% |
| ` of` | -0.0000 | 100.0% |
| ` grass` | -0.0000 | 100.0% |
| `,` | -0.8242 | 43.9% |
| ` a` | -0.7104 | 49.1% |
| ` reminder` | -0.3012 | 74.0% |
| ` of` | -0.5761 | 56.2% |
| ` the` | -0.5265 | 59.1% |
| ` delicate` | -1.9362 | 14.4% |
| ` equations` | -0.7571 | 46.9% |
| ` of` | -0.7673 | 46.4% |
| ` time` | -1.0988 | 33.3% |
| ` and` | -0.1441 | 86.6% |
| ` light` | -0.4314 | 65.0% |
| ` that` | -0.8434 | 43.0% |
| ` govern` | -1.6522 | 19.2% |
| ` our` | -0.1342 | 87.4% |
| ` perception` | -1.0096 | 36.4% |
| `.` | -1.1589 | 31.4% |
| ` It` | -1.4645 | 23.1% |
| `’s` | -0.2092 | 81.1% |
| ` a` | -0.3319 | 71.8% |
| ` taste` | -0.2861 | 75.1% |
| ` that` | -0.1581 | 85.4% |
| ` dances` | -1.0806 | 33.9% |
| ` on` | -0.2627 | 76.9% |
| ` the` | -0.0111 | 98.9% |
| ` tongue` | -0.3916 | 67.6% |
| `,` | -0.3998 | 67.0% |
| ` a` | -1.7288 | 17.7% |
| ` fleeting` | -1.9073 | 14.8% |
| ` theorem` | -1.7049 | 18.2% |
| ` of` | -0.4062 | 66.6% |
| ` dusk` | -0.8762 | 41.6% |
| ` that` | -0.7228 | 48.5% |
| ` van` | -1.1890 | 30.5% |
| `ishes` | -0.0000 | 100.0% |
| ` before` | -0.8437 | 43.0% |
| ` it` | -0.7310 | 48.1% |
| ` can` | -0.0110 | 98.9% |
| ` be` | -0.0808 | 92.2% |
| ` fully` | -0.0550 | 94.6% |
| ` grasp` | -0.8652 | 42.1% |
| `ed` | 0.0000 | 100.0% |
| `,` | -0.3712 | 69.0% |
| ` leaving` | -0.5643 | 56.9% |
| ` only` | -0.2791 | 75.6% |
| ` the` | -0.4349 | 64.7% |
| ` echo` | -1.0615 | 34.6% |
| ` of` | -0.0000 | 100.0% |
| ` its` | -0.0204 | 98.0% |
| ` brilliance` | -0.7689 | 46.4% |
| ` behind` | -0.5281 | 59.0% |
| `.` | -0.0153 | 98.5% |

**Perplexity:** `1.85`

---